# 02 · Silver — tipado, deduplicación y reglas DQ01–DQ12

Valida el lote de Bronze con `config/rules.yaml`. VALID y WARNING van a `storage/silver/`, INVALID a `storage/quarantine/` con el motivo. Los hallazgos quedan en `gold.dq_errors`.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src import pipeline
from src.common import db, lake
from src.config import get_settings, load_pipeline

engine = db.get_engine()
settings, cfg = get_settings(), load_pipeline()

In [ ]:
# Lote a procesar: None = el último registrado en gold.dq_run_metrics
BATCH_ID = None
BATCH_ID = BATCH_ID or pipeline.latest_batch_id(engine)
BATCH_ID

In [ ]:
result = pipeline.run_silver(engine, BATCH_ID)
print(f'VALID={result.valid}  WARNING={result.warning}  INVALID={result.invalid}  DQ Score={result.dq_score}%')

In [ ]:
result.findings.groupby(['rule_id', 'action']).size().rename('hallazgos').to_frame()

In [ ]:
result.quarantine[['timestamp', 'field_id', 'source', '_rejection_reasons']].head(20)